# 0. Ingesta, integraci?n y limpieza

Consolida archivos mensuales desde julio de 2022 hasta agosto de 2026. Admite CSV delimitado por punto y coma y Excel. Genera auditor?a de cobertura, normaliza campos y conserva una bandera de relevancia revisable.

**Limitaciones del origen:** los esquemas examinados no incluyen latitud/longitud ni hora; `Fecha` aparece normalmente al nivel de d?a. El cuaderno no inventa coordenadas ni componentes temporales ausentes. La relevancia usa reglas transparentes editables, que deben validarse con expertos antes de usar alertas.

In [ ]:
from pathlib import Path
import re, unicodedata
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/raw").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
INICIO, FIN = pd.Timestamp("2022-07-01"), pd.Timestamp("2026-08-31")
MESES = {"enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,"julio":7,"agosto":8,"septiembre":9,"setiembre":9,"octubre":10,"noviembre":11,"diciembre":12}


## Lectura tolerante y control de cobertura

La fecha del registro es la fuente de verdad; el nombre del archivo se usa para auditar y rescatar filas sin fecha. Los archivos se detectan por a?o en el nombre de carpeta, incluyendo extensiones y nombres irregulares.

In [ ]:
def leer_archivo(path):
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    ultimo = None
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            df = pd.read_csv(path, sep=None, engine="python", encoding=enc, low_memory=False)
            if len(df.columns) == 1 and ";" in str(df.columns[0]):
                df = pd.read_csv(path, sep=";", encoding=enc, low_memory=False)
            return df
        except Exception as exc: ultimo = exc
    raise ultimo

def mes_desde_nombre(nombre):
    s = nombre.lower()
    for m, n in MESES.items():
        if re.search(rf"{m}[_-]?(20\d{{2}})", s): return n
    return None

frames, auditoria = [], []
for path in sorted(RAW.glob("data20[2-2][0-9]/*")):
    if path.suffix.lower() not in {".csv", ".xlsx", ".xls"}: continue
    try:
        df = leer_archivo(path)
        df.columns = [str(c).strip().lstrip("\ufeff") for c in df.columns]
        df["archivo_origen"] = str(path.relative_to(ROOT))
        df["anio_archivo"] = int(path.parent.name.replace("data", ""))
        if "Fecha" not in df.columns: raise ValueError("No existe columna Fecha")
        # Formato com?n d/m/yyyy; dayfirst cubre variantes del origen.
        df["fecha"] = pd.to_datetime(df["Fecha"], errors="coerce", dayfirst=True)
        missing = df["fecha"].isna()
        m = mes_desde_nombre(path.stem)
        if missing.any() and m:
            df.loc[missing, "fecha"] = pd.to_datetime(
                dict(year=df.loc[missing,"anio_archivo"], month=m, day=1), errors="coerce")
        auditoria.append({"archivo":str(path.relative_to(ROOT)),"filas":len(df),"fechas_invalidas":int(df["fecha"].isna().sum()),"columnas":", ".join(df.columns)})
        frames.append(df)
    except Exception as exc:
        auditoria.append({"archivo":str(path.relative_to(ROOT)),"error":str(exc)})

if not frames: raise FileNotFoundError(f"No se encontraron datos en {RAW}")
raw = pd.concat(frames, ignore_index=True, sort=False)
auditoria = pd.DataFrame(auditoria)
auditoria.to_csv(OUT / "auditoria_ingesta.csv", index=False, encoding="utf-8-sig")
raw = raw.loc[raw.fecha.between(INICIO, FIN)].copy()
print(f"Filas dentro del periodo: {len(raw):,}; archivos le?dos: {len(frames)}")


In [ ]:
# Normalizaci?n de categor?as (sin alterar el texto fuente).
for col in ["provincia", "Canton", "Parroquia", "Servicio", "Subtipo"]:
    if col not in raw: raw[col] = pd.NA
    raw[col] = raw[col].astype("string").str.strip().replace({"":pd.NA, "nan":pd.NA})
raw["provincia"] = raw["provincia"].str.upper()
raw["Canton"] = raw["Canton"].str.upper()
raw["fecha"] = pd.to_datetime(raw["fecha"], errors="coerce")
raw = raw.dropna(subset=["fecha"]).copy()
raw["anio"] = raw.fecha.dt.year
raw["mes"] = raw.fecha.dt.month
raw["dia"] = raw.fecha.dt.day
raw["hora"] = raw.fecha.dt.hour  # nulo si la fuente no tiene hora

# Reglas de relevancia iniciales: transparentes, modificables y deliberadamente amplias.
def normalizar(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")
PATRONES_RIESGO = (
    "robo", "asalto", "secuestro", "extorsion", "homicidio", "asesinato", "violencia",
    "agresion", "arma", "disparo", "accidente de transito", "siniestro de transito",
    "incendio", "inundacion", "deslizamiento", "sismo", "ahogamiento", "desaparecid",
    "intoxicacion", "emergencia medica", "rescate", "violacion", "acoso"
)
texto = (raw["Servicio"].fillna("") + " " + raw["Subtipo"].fillna("")).map(normalizar)
raw["relevante_turismo"] = texto.map(lambda s: any(p in s for p in PATRONES_RIESGO))

# Las columnas no existen en el origen conocido. Si llegan en nuevos archivos, validar l?mites Ecuador.
for c in ("latitud", "longitud"):
    if c not in raw: raw[c] = pd.NA
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
coord_valida = raw.latitud.between(-5.5, 1.7) & raw.longitud.between(-92.5, -75.0)
raw["coordenadas_validas"] = coord_valida.fillna(False)
print("% coordenadas v?lidas:", round(raw.coordenadas_validas.mean()*100, 3))
print("% registros marcados relevantes:", round(raw.relevante_turismo.mean()*100, 2))


## Cobertura, salidas y limitaciones

El agregado conserva el total de incidentes y el total marcado relevante para que las reglas puedan revisarse. Para an?lisis de riesgo tur?stico se filtra `relevante_turismo`, pero no debe interpretarse como una clasificaci?n oficial. No se imputa clima ni hora: requieren fuentes adicionales. La validaci?n geogr?fica usa l?mites amplios de Ecuador continental; Gal?pagos requiere un rango separado si se incorporan coordenadas.

In [ ]:
# Calendario mensual observado vs. esperado. Un mes sin archivo no equivale a cero incidentes.
archivos = pd.DataFrame(auditoria)
meses_esperados = pd.period_range("2022-07", "2026-08", freq="M")
meses_observados = pd.PeriodIndex(raw.fecha.dt.to_period("M").dropna().unique(), freq="M")
calendario = pd.DataFrame({"mes":meses_esperados})
calendario["observado"] = calendario.mes.isin(meses_observados)
calendario.to_csv(OUT / "cobertura_mensual.csv", index=False, encoding="utf-8-sig")
print(calendario.loc[~calendario.observado])
raw.to_parquet(OUT / "incidentes_integrados.parquet", index=False)
raw.to_csv(OUT / "incidentes_integrados.csv.gz", index=False, compression="gzip", encoding="utf-8-sig")
relevantes = raw.loc[raw.relevante_turismo].copy()
relevantes.to_parquet(OUT / "incidentes_relevantes.parquet", index=False)
print("Guardados:", OUT)
